In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
pd.set_option("display.max_columns", None)

In [10]:
df = pd.read_csv('../raw_data/investments_VC.csv', encoding='latin1', low_memory=False)
print(df.shape)

(54294, 39)


In [14]:
# drops all rows with more than 50% missing data
row_missing_pct = df.isna().mean(axis=1).mul(100)
df = df[row_missing_pct <= 50]

print(f"New shape: {df.shape}")

New shape: (49438, 39)


In [ ]:
# printing missing values
missing = pd.DataFrame({
'missing_count': df.isna().sum(),
'missing_pct': df.isna().mean().mul(100).round(2),
'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

print(missing)

                      missing_count  missing_pct    dtype
state_code                    19277        38.99   object
founded_year                  10956        22.16  float64
founded_quarter               10956        22.16   object
founded_month                 10956        22.16   object
founded_at                    10884        22.02   object
city                           6116        12.37   object
country_code                   5273        10.67   object
region                         5273        10.67   object
 market                        3968         8.03   object
category_list                  3961         8.01   object
homepage_url                   3449         6.98   object
status                         1314         2.66   object
round_C                           0         0.00  float64
post_ipo_debt                     0         0.00  float64
secondary_market                  0         0.00  float64
product_crowdfunding              0         0.00  float64
round_A       

In [17]:
cols_to_drop = [
'city',
'founded_quarter', # can be derived from founded_at
'founded_month', # can be derived from founded_at
'founded_year', # can be derived from founded_at
'homepage_url',
'name',
]

df = df.drop(columns=cols_to_drop)
print(f"New shape: {df.shape}")

New shape: (49438, 33)


In [27]:
# Check current types
print(df.dtypes)

permalink                       object
category_list                   object
 market                         object
 funding_total_usd              object
status                          object
country_code                    object
state_code                      object
region                          object
funding_rounds                 float64
founded_at              datetime64[ns]
first_funding_at        datetime64[ns]
last_funding_at         datetime64[ns]
seed                           float64
venture                        float64
equity_crowdfunding            float64
undisclosed                    float64
convertible_note               float64
debt_financing                 float64
angel                          float64
grant                          float64
private_equity                 float64
post_ipo_equity                float64
post_ipo_debt                  float64
secondary_market               float64
product_crowdfunding           float64
round_A                  

In [26]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = df[col].str.strip()   # remove whitespace
    df[col] = df[col].str.lower()   # standardise case

print("String columns cleaned")

String columns cleaned


In [29]:
# This fixes the column NAMES/HEADERS
df.columns = df.columns.str.strip()

In [30]:
print(df.dtypes)

permalink                       object
category_list                   object
market                          object
funding_total_usd               object
status                          object
country_code                    object
state_code                      object
region                          object
funding_rounds                 float64
founded_at              datetime64[ns]
first_funding_at        datetime64[ns]
last_funding_at         datetime64[ns]
seed                           float64
venture                        float64
equity_crowdfunding            float64
undisclosed                    float64
convertible_note               float64
debt_financing                 float64
angel                          float64
grant                          float64
private_equity                 float64
post_ipo_equity                float64
post_ipo_debt                  float64
secondary_market               float64
product_crowdfunding           float64
round_A                  

In [32]:
# Convert all date columns from object to datetime
date_cols = ['founded_at', 'first_funding_at', 'last_funding_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# funding_total_usd may have come in as object (string) due to commas/symbols
df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')

# Verify the changes
print("\nAfter conversion:")
print(df[date_cols + ['founded_year', 'funding_total_usd']].dtypes)


After conversion:
founded_at           datetime64[ns]
first_funding_at     datetime64[ns]
last_funding_at      datetime64[ns]
founded_year                float64
funding_total_usd           float64
dtype: object


In [37]:
cols_to_drop = [
'founded_year', # can be derived from founded_at
]

df = df.drop(columns=cols_to_drop)

In [38]:
print(df.shape)

(49438, 33)


In [ ]:
# How many companies founded before 2000
print(f"Companies founded before 2000: {(df['founded_at'] < '2000-01-01').sum()}")

Companies founded before 2000: 3730


In [45]:
df = df[(df['founded_at'] >= '2000-01-01') | (df['founded_at'].isna())]
print(f"New shape: {df.shape}")

New shape: (45708, 33)
